## Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.tree import DecisionTreeRegressor, export_text

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host= 
    # 'processing.processing.data.production.internal',
    # 'processing-2.processing.data.production.internal',
    'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/geo-consistency'
print(local_dataset)

/Users/E2074/analytics/customer/segmentation/segment-appography
/Users/E2074/local-datasets/customer/geo-consistency


## Datasets & Parameter

In [5]:
city = 'Bangalore'
date_str = '20250316'

In [6]:
base_query = f"""

with funnel as (
    
    select 
        -- week_start_date,
        customerid as customer_id,
        SUM(cast(ao_session as int)) as ao_session,
        SUM(cast(fe_session as int)) as fe_session,
        SUM(cast(rr_session as int)) as rr_session
    from 
        hive.experiments_internal.customer_funnel_last90days_20250317
    where 
        week_start_date = '20250310'
    group by 1
    ),
    
    appo as (
    
    select 
        userid,
        deliveryFood,
        deliveryGrocery,
        driverApp,
        fantasySports,
        financeInvestment,
        gaming,
        health,
        homeSecurity,
        office,
        ott,
        streamingMusic,
        travelBookings,
        travelRidehailing
    
    from 
        reports.sql_ingestion_customer_appography_category_view 
    where 
        yyyymmdd = '20250330'
    ),

    segment as (

    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    )
    
    select
        funnel.customer_id,
        segment,
        deliveryFood,
        deliveryGrocery,
        driverApp,
        fantasySports,
        financeInvestment,
        gaming,
        health,
        homeSecurity,
        office,
        ott,
        streamingMusic,
        travelBookings,
        travelRidehailing,
        ao_session,
        fe_session,
        rr_session
    from 
        segment  
    join 
        appo
        on appo.userid = segment.customer_id
    left join 
        funnel
        on funnel.customer_id = segment.customer_id

"""

# df_base_query = pd.read_sql(base_query, connection)

In [8]:
print(df_base_query.shape)
display(df_base_query.head())

(4110361, 18)


,customer_id,segment,deliveryFood,deliveryGrocery,driverApp,fantasySports,financeInvestment,gaming,health,homeSecurity,office,ott,streamingMusic,travelBookings,travelRidehailing,ao_session,fe_session,rr_session
0,6291deea4480f6e8dbc5de3d,1-DAILY,1,1,0,0,1,1,1,1,1,1,1,1,1,251,154,170
1,639c7da3ad27b5845f899d1b,5-INACTIVE,1,1,0,0,1,0,0,0,1,1,1,1,1,0,0,0
2,63be17aafbf87ea29744f366,4-OTHER,1,1,0,0,1,1,1,1,1,1,1,1,1,1,1,1
3,670d0f854618974072145131,3-MONTHLY,0,0,0,0,0,0,0,0,1,0,1,0,1,18,14,11
4,5d0bba711c64cc470f65f803,3-MONTHLY,1,1,0,1,1,0,0,1,1,1,1,1,1,74,59,51


In [28]:
# df_base_query.to_parquet('prunning-dump.parquet', index=False)

In [7]:
df_base_query = pd.read_parquet('prunning-dump.parquet')
df_base_query.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4110361 entries, 0 to 4110360
Data columns (total 18 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   customer_id        object
 1   segment            object
 2   deliveryFood       int64 
 3   deliveryGrocery    int64 
 4   driverApp          int64 
 5   fantasySports      int64 
 6   financeInvestment  int64 
 7   gaming             int64 
 8   health             int64 
 9   homeSecurity       int64 
 10  office             int64 
 11  ott                int64 
 12  streamingMusic     int64 
 13  travelBookings     int64 
 14  travelRidehailing  int64 
 15  ao_session         int64 
 16  fe_session         int64 
 17  rr_session         int64 
dtypes: int64(16), object(2)
memory usage: 564.5+ MB


In [9]:
print(df_base_query.shape)
df_base_query.head(5)

(4110361, 18)


,customer_id,segment,deliveryFood,deliveryGrocery,driverApp,fantasySports,financeInvestment,gaming,health,homeSecurity,office,ott,streamingMusic,travelBookings,travelRidehailing,ao_session,fe_session,rr_session
0,6291deea4480f6e8dbc5de3d,1-DAILY,1,1,0,0,1,1,1,1,1,1,1,1,1,251,154,170
1,639c7da3ad27b5845f899d1b,5-INACTIVE,1,1,0,0,1,0,0,0,1,1,1,1,1,0,0,0
2,63be17aafbf87ea29744f366,4-OTHER,1,1,0,0,1,1,1,1,1,1,1,1,1,1,1,1
3,670d0f854618974072145131,3-MONTHLY,0,0,0,0,0,0,0,0,1,0,1,0,1,18,14,11
4,5d0bba711c64cc470f65f803,3-MONTHLY,1,1,0,1,1,0,0,1,1,1,1,1,1,74,59,51


In [10]:
def data_agg(df, column_list):
    df_temp = df\
                .groupby(column_list)\
                .agg(customers = ('customer_id', 'nunique'),
                     ao_session = ('ao_session', 'sum'),
                     fe_session = ('fe_session', 'sum'),
                     rr_session = ('rr_session', 'sum')
                    )\
                .reset_index()\
                .sort_values(by=column_list, ascending=False)

    df_temp['total_customer'] = df_base_query.customer_id.nunique()
    df_temp['cux_distr'] = round( df_temp['customers']*100.00/df_temp['total_customer'] ,0)
    df_temp['fe2rr'] = round( df_temp['rr_session']*100.00/df_temp['fe_session'] ,0)
    df_temp['ao2fe'] = round( df_temp['fe_session']*100.00/df_temp['ao_session'] ,0)
    df_temp['ao2rr'] = round( df_temp['rr_session']*100.00/df_temp['ao_session'] ,0)
    
    
    return df_temp

In [11]:
df_analysis = data_agg(df_base_query, ['travelRidehailing', 'deliveryFood', 'deliveryGrocery', 'travelBookings'])
df_analysis

,travelRidehailing,deliveryFood,deliveryGrocery,travelBookings,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
15,1,1,1,1,1289720,26101172,21885470,15877072,4110361,31.0,73.0,84.0,61.0
14,1,1,1,0,372754,8345580,6875933,5040633,4110361,9.0,73.0,82.0,60.0
13,1,1,0,1,341141,4697872,3865916,2588676,4110361,8.0,67.0,82.0,55.0
12,1,1,0,0,126274,2301967,1854224,1321035,4110361,3.0,71.0,81.0,57.0
11,1,0,1,1,132369,1974000,1637011,974630,4110361,3.0,60.0,83.0,49.0
10,1,0,1,0,94066,1692770,1374577,907997,4110361,2.0,66.0,81.0,54.0
9,1,0,0,1,311653,4144182,3386932,1984136,4110361,8.0,59.0,82.0,48.0
8,1,0,0,0,198882,3307498,2639115,1707402,4110361,5.0,65.0,80.0,52.0
7,0,1,1,1,119076,1760530,1447975,932621,4110361,3.0,64.0,82.0,53.0
6,0,1,1,0,100633,1729871,1393493,979373,4110361,2.0,70.0,81.0,57.0


In [12]:
df_analysis = data_agg(df_base_query, ['travelRidehailing', 'deliveryFood', 'deliveryGrocery'])
df_analysis

,travelRidehailing,deliveryFood,deliveryGrocery,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
7,1,1,1,1662474,34446752,28761403,20917705,4110361,40.0,73.0,83.0,61.0
6,1,1,0,467415,6999839,5720140,3909711,4110361,11.0,68.0,82.0,56.0
5,1,0,1,226435,3666770,3011588,1882627,4110361,6.0,63.0,82.0,51.0
4,1,0,0,510535,7451680,6026047,3691538,4110361,12.0,61.0,81.0,50.0
3,0,1,1,219709,3490401,2841468,1911994,4110361,5.0,67.0,81.0,55.0
2,0,1,0,176836,2356007,1885580,1217445,4110361,4.0,65.0,80.0,52.0
1,0,0,1,160315,2211212,1771236,1080979,4110361,4.0,61.0,80.0,49.0
0,0,0,0,686642,8856291,6945009,4102136,4110361,17.0,59.0,78.0,46.0


In [51]:
df_analysis.to_clipboard(index=False)

In [13]:
df_analysis = data_agg(df_base_query, ['travelRidehailing', 'deliveryFood', 'deliveryGrocery','financeInvestment'])
df_analysis

,travelRidehailing,deliveryFood,deliveryGrocery,financeInvestment,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
15,1,1,1,1,1081763,22008283,18444700,13575556,4110361,26.0,74.0,84.0,62.0
14,1,1,1,0,580711,12438469,10316703,7342149,4110361,14.0,71.0,83.0,59.0
13,1,1,0,1,261864,3644085,2997264,2051570,4110361,6.0,68.0,82.0,56.0
12,1,1,0,0,205551,3355754,2722876,1858141,4110361,5.0,68.0,81.0,55.0
11,1,0,1,1,94161,1426933,1182687,731722,4110361,2.0,62.0,83.0,51.0
10,1,0,1,0,132274,2239837,1828901,1150905,4110361,3.0,63.0,82.0,51.0
9,1,0,0,1,190777,2494433,2039809,1238726,4110361,5.0,61.0,82.0,50.0
8,1,0,0,0,319758,4957247,3986238,2452812,4110361,8.0,62.0,80.0,49.0
7,0,1,1,1,89232,1311632,1075240,723667,4110361,2.0,67.0,82.0,55.0
6,0,1,1,0,130477,2178769,1766228,1188327,4110361,3.0,67.0,81.0,55.0


In [38]:
# 'driverApp',	'fantasySports',	'gaming','health',	'homeSecurity',	'office',	'ott',	'streamingMusic'
df_analysis = data_agg(df_base_query, [ 'travelRidehailing', 'deliveryFood', 'deliveryGrocery', 'driverApp'	])
df_analysis

# 'travelRidehailing', 'deliveryFood', 'deliveryGrocery','financeInvestment'
# 'driverApp', 'office', 'homeSecurity', 'health'


# 'fantasySports',	'gaming', 'ott', 'streamingMusic'

,travelRidehailing,deliveryFood,deliveryGrocery,driverApp,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
15,1,1,1,1,192222,3143602,2602959,1716441,4110361,5.0,66.0,83.0,55.0
14,1,1,1,0,1470252,31303150,26158444,19201264,4110361,36.0,73.0,84.0,61.0
13,1,1,0,1,60978,812361,662710,407105,4110361,1.0,61.0,82.0,50.0
12,1,1,0,0,406437,6187478,5057430,3502606,4110361,10.0,69.0,82.0,57.0
11,1,0,1,1,35961,506373,413320,233801,4110361,1.0,57.0,82.0,46.0
10,1,0,1,0,190474,3160397,2598268,1648826,4110361,5.0,63.0,82.0,52.0
9,1,0,0,1,69386,924928,755266,410707,4110361,2.0,54.0,82.0,44.0
8,1,0,0,0,441149,6526752,5270781,3280831,4110361,11.0,62.0,81.0,50.0
7,0,1,1,1,36024,458364,368893,227782,4110361,1.0,62.0,80.0,50.0
6,0,1,1,0,183685,3032037,2472575,1684212,4110361,4.0,68.0,82.0,56.0


In [15]:
def decision_tree(df, grouper):
    # 2. Define Features (App Category Flags) and Target (FE2R)
    X = df[grouper]
    y = df['fe2rr']

    # 3. Create Decision Tree Regressor
    tree_model = DecisionTreeRegressor(
        max_depth=5,     # limit depth so that tree is interpretable
        min_samples_leaf=2,  # minimum samples in each leaf
        random_state=42
    )
    
    # 4. Fit the model
    tree_model.fit(X, y)
    
    # 5. Visualize tree rules
    tree_rules = export_text(tree_model, feature_names=list(X.columns))
    print(tree_rules)

In [37]:
decision_tree(df_analysis, ['travelRidehailing', 'deliveryFood', 'deliveryGrocery','financeInvestment'])

|--- deliveryFood <= 0.50
|   |--- travelRidehailing <= 0.50
|   |   |--- deliveryGrocery <= 0.50
|   |   |   |--- value: [59.00]
|   |   |--- deliveryGrocery >  0.50
|   |   |   |--- value: [60.50]
|   |--- travelRidehailing >  0.50
|   |   |--- deliveryGrocery <= 0.50
|   |   |   |--- value: [61.50]
|   |   |--- deliveryGrocery >  0.50
|   |   |   |--- value: [62.50]
|--- deliveryFood >  0.50
|   |--- travelRidehailing <= 0.50
|   |   |--- deliveryGrocery <= 0.50
|   |   |   |--- value: [64.50]
|   |   |--- deliveryGrocery >  0.50
|   |   |   |--- value: [67.00]
|   |--- travelRidehailing >  0.50
|   |   |--- deliveryGrocery <= 0.50
|   |   |   |--- value: [68.00]
|   |   |--- deliveryGrocery >  0.50
|   |   |   |--- value: [72.50]



In [38]:
decision_tree(data_agg(df_base_query, ['travelRidehailing', 'deliveryFood', 'deliveryGrocery']), 
              ['travelRidehailing', 'deliveryFood', 'deliveryGrocery'])

|--- deliveryFood <= 0.50
|   |--- deliveryGrocery <= 0.50
|   |   |--- value: [60.00]
|   |--- deliveryGrocery >  0.50
|   |   |--- value: [62.00]
|--- deliveryFood >  0.50
|   |--- travelRidehailing <= 0.50
|   |   |--- value: [66.00]
|   |--- travelRidehailing >  0.50
|   |   |--- value: [70.50]



In [39]:
decision_tree(data_agg(df_base_query, ['travelRidehailing', 'deliveryFood', 'deliveryGrocery', 'travelBookings']), 
              ['travelRidehailing', 'deliveryFood', 'deliveryGrocery','travelBookings'])

|--- deliveryFood <= 0.50
|   |--- travelBookings <= 0.50
|   |   |--- travelRidehailing <= 0.50
|   |   |   |--- value: [63.00]
|   |   |--- travelRidehailing >  0.50
|   |   |   |--- value: [65.50]
|   |--- travelBookings >  0.50
|   |   |--- travelRidehailing <= 0.50
|   |   |   |--- value: [56.00]
|   |   |--- travelRidehailing >  0.50
|   |   |   |--- value: [59.50]
|--- deliveryFood >  0.50
|   |--- travelRidehailing <= 0.50
|   |   |--- travelBookings <= 0.50
|   |   |   |--- value: [69.00]
|   |   |--- travelBookings >  0.50
|   |   |   |--- value: [62.50]
|   |--- travelRidehailing >  0.50
|   |   |--- deliveryGrocery <= 0.50
|   |   |   |--- value: [69.00]
|   |   |--- deliveryGrocery >  0.50
|   |   |   |--- value: [73.00]



In [36]:
decision_tree(df_analysis, [  'office', 'homeSecurity', 'health'	])

|--- homeSecurity <= 0.50
|   |--- office <= 0.50
|   |   |--- health <= 0.50
|   |   |   |--- value: [59.50]
|   |   |--- health >  0.50
|   |   |   |--- value: [62.00]
|   |--- office >  0.50
|   |   |--- health <= 0.50
|   |   |   |--- value: [65.00]
|   |   |--- health >  0.50
|   |   |   |--- value: [69.50]
|--- homeSecurity >  0.50
|   |--- office <= 0.50
|   |   |--- health <= 0.50
|   |   |   |--- value: [69.00]
|   |   |--- health >  0.50
|   |   |   |--- value: [72.50]
|   |--- office >  0.50
|   |   |--- health <= 0.50
|   |   |   |--- value: [74.00]
|   |   |--- health >  0.50
|   |   |   |--- value: [80.00]



## Analysis

In [40]:
grouper = ['segment', 'travelRidehailing', 'deliveryFood', 'deliveryGrocery','financeInvestment']

df_analysis = data_agg(df_base_query, grouper).sort_values(by='segment', ascending=True)
df_analysis

,segment,travelRidehailing,deliveryFood,deliveryGrocery,financeInvestment,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
0,1-DAILY,0,0,0,0,3770,585329,453556,386133,4110361,0.0,85.0,77.0,66.0
15,1-DAILY,1,1,1,1,32684,4532332,3744350,3364108,4110361,1.0,90.0,83.0,74.0
14,1-DAILY,1,1,1,0,18889,2759437,2262402,1982556,4110361,0.0,88.0,82.0,72.0
13,1-DAILY,1,1,0,1,3311,465608,376299,331292,4110361,0.0,88.0,81.0,71.0
12,1-DAILY,1,1,0,0,3823,564185,450637,390843,4110361,0.0,87.0,80.0,69.0
11,1-DAILY,1,0,1,1,1241,179757,146071,124469,4110361,0.0,85.0,81.0,69.0
10,1-DAILY,1,0,1,0,2286,345555,279972,233827,4110361,0.0,84.0,81.0,68.0
9,1-DAILY,1,0,0,1,1773,267001,212840,179697,4110361,0.0,84.0,80.0,67.0
1,1-DAILY,0,0,0,1,807,118426,92836,80424,4110361,0.0,87.0,78.0,68.0
7,1-DAILY,0,1,1,1,998,133002,107877,98150,4110361,0.0,91.0,81.0,74.0


In [39]:
grouper = ['travelRidehailing', 'deliveryFood', 'deliveryGrocery', 'driverApp']

df_analysis = data_agg(df_base_query, grouper)
df_analysis






,travelRidehailing,deliveryFood,deliveryGrocery,driverApp,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
15,1,1,1,1,192222,3143602,2602959,1716441,4110361,5.0,66.0,83.0,55.0
14,1,1,1,0,1470252,31303150,26158444,19201264,4110361,36.0,73.0,84.0,61.0
13,1,1,0,1,60978,812361,662710,407105,4110361,1.0,61.0,82.0,50.0
12,1,1,0,0,406437,6187478,5057430,3502606,4110361,10.0,69.0,82.0,57.0
11,1,0,1,1,35961,506373,413320,233801,4110361,1.0,57.0,82.0,46.0
10,1,0,1,0,190474,3160397,2598268,1648826,4110361,5.0,63.0,82.0,52.0
9,1,0,0,1,69386,924928,755266,410707,4110361,2.0,54.0,82.0,44.0
8,1,0,0,0,441149,6526752,5270781,3280831,4110361,11.0,62.0,81.0,50.0
7,0,1,1,1,36024,458364,368893,227782,4110361,1.0,62.0,80.0,50.0
6,0,1,1,0,183685,3032037,2472575,1684212,4110361,4.0,68.0,82.0,56.0


In [42]:
grouper = ['segment', 'travelRidehailing', 'deliveryFood', 'deliveryGrocery']

df_analysis = data_agg(df_base_query, grouper).sort_values(by=['segment','travelRidehailing', 'deliveryFood', 'deliveryGrocery'], ascending=[True, False, False, False])
df_analysis

,segment,travelRidehailing,deliveryFood,deliveryGrocery,customers,ao_session,fe_session,rr_session,total_customer,cux_distr,fe2rr,ao2fe,ao2rr
7,1-DAILY,1,1,1,51573,7291769,6006752,5346664,4110361,1.0,89.0,82.0,73.0
6,1-DAILY,1,1,0,7134,1029793,826936,722135,4110361,0.0,87.0,80.0,70.0
5,1-DAILY,1,0,1,3527,525312,426043,358296,4110361,0.0,84.0,81.0,68.0
4,1-DAILY,1,0,0,5950,928011,736540,614117,4110361,0.0,83.0,79.0,66.0
3,1-DAILY,0,1,1,3051,427711,345279,311014,4110361,0.0,90.0,81.0,73.0
2,1-DAILY,0,1,0,1654,236648,187487,166347,4110361,0.0,89.0,79.0,70.0
1,1-DAILY,0,0,1,1390,201508,159761,138643,4110361,0.0,87.0,79.0,69.0
0,1-DAILY,0,0,0,4577,703755,546392,466557,4110361,0.0,85.0,78.0,66.0
15,2-WEEKLY,1,1,1,135896,8275386,6847059,5448093,4110361,3.0,80.0,83.0,66.0
14,2-WEEKLY,1,1,0,22276,1349253,1085281,851593,4110361,1.0,78.0,80.0,63.0


### All category
